## 🏗️ Mini Project — Naive Bayes

``Email Spam Classifier (Simplified Numeric Version)``

In [11]:
import numpy as np

np.random.seed(30)
n_emails = 300

# Features: word frequency counts (simplified numeric proxies)
exclamation_marks = np.random.poisson(2, n_emails)
capital_word_ratio = np.random.beta(2, 5, n_emails)  # 0-1 range
suspicious_word_count = np.random.poisson(1.5, n_emails)
email_length = np.random.normal(150, 50, n_emails).clip(20, 500)

# True relationship: high exclamation, high caps ratio, suspicious words → spam
spam_score = (0.8*exclamation_marks + 5*capital_word_ratio + 
              1.2*suspicious_word_count - 0.005*email_length - 2)
spam_prob = 1/(1+np.exp(-spam_score))
is_spam = np.random.binomial(1, spam_prob)

X = np.column_stack([exclamation_marks, capital_word_ratio, suspicious_word_count, email_length])
y = is_spam

print(f"Spam rate: {y.mean():.1%}")

Spam rate: 82.0%


In [12]:
class GuassianNaiveBayesScratch:
    def __init__(self):
        self.class_priors = {}
        self.means = {}
        self.variances = {}
        self.classes = None

    def fit(self, X, y):
        self.classes = np.unique(y)
        n_samples = X.shape[0]

        for c in self.classes:
            X_c = X[y == c]

            self.class_priors[c] = len(X_c)/n_samples

            self.means[c] = X_c.mean(axis =0)
            self.variances[c] = X_c.var(axis =0)

    def gaussian_probability(self, x, mean, var):
        eps = 1e-9

        numerator = np.exp(-((x - mean) ** 2) / (2* var + eps))
        denominator = np.sqrt(2* np.pi * var + eps)
        return numerator /denominator

    def predict_single(self, x):
        class_scores = {}

        for c in self.classes:
            score = np.log(self.class_priors[c])

            probabilities = self.gaussian_probability(x, self.means[c], self.variances[c])

            score += np.sum(np.log(probabilities + 1e-9))
            
            class_scores[c] = score

        return max(class_scores, key= class_scores.get)

    def predict(self, X):
        return np.array([self.predict_single(x) for x in X])

In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

In [18]:
nb_scratch = GuassianNaiveBayesScratch()

nb_scratch.fit(X_train, y_train)

pred = nb_scratch.predict(X_test)

accuracy = np.mean(pred == y_test)

print(f'Accuracy: {accuracy:.2%}')

print(nb_scratch.class_priors)
print(nb_scratch.means)
print(nb_scratch.variances)

Accuracy: 78.33%
{np.int32(0): 0.1625, np.int32(1): 0.8375}
{np.int32(0): array([  1.28205128,   0.24704247,   0.61538462, 155.41826617]), np.int32(1): array([  2.13930348,   0.3096544 ,   1.77114428, 145.44927479])}
{np.int32(0): array([9.71729126e-01, 1.96110260e-02, 5.95660750e-01, 1.56080660e+03]), np.int32(1): array([2.26915175e+00, 2.55882078e-02, 1.80832158e+00, 2.87992815e+03])}


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

for model in [LogisticRegression(), KNeighborsClassifier(), GaussianNB()]:

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)

    accuracy = model.score(X_test, y_test)

    print(f'Model: {model} - Accuracy: {accuracy:.2%}')

Model: LogisticRegression() - Accuracy: 75.00%
Model: KNeighborsClassifier() - Accuracy: 76.67%
Model: GaussianNB() - Accuracy: 78.33%
